<a href="https://colab.research.google.com/github/fazmina11/fazmina-codeboosters-2026/blob/main/Day3/Day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")
print("NumPy:", np.__version__)
print("Requests:", requests.__version__)

All libraries imported successfully
NumPy: 2.0.2
Requests: 2.32.4


In [3]:

raw_df=pd.read_csv("messy_sales_data.csv")
print(f"Raw data loaded {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
print(f"Columns:{raw_df.columns.tolist()}")
raw_df
##raw_df.head()
#print("First 5 Rows\n", raw_df.head().to_string(index=False))

Raw data loaded 30 rows, 9 columns
Columns:['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
5,1006,kiran mehta,Mouse,Accessories,10.0,800,07-01-2024,Pune,Sunita Rao
6,1007,Deepak Singh,Headphones,Electronics,2.0,3500,2024-01-12,Hyderabad,Ravi Kumar
7,1008,NaN,Webcam,Accessories,1.0,2500,2024-01-13,Mumbai,Anil Sharma
8,1009,Ananya Das,Laptop,Electronics,1.0,45000,2024-01-15,Kolkata,Sunita Rao
9,1010,Vikram Iyer,Keyboard,Accessories,5.0,1200,2024-01-15,Chennai,Ravi Kumar


In [4]:

print("="*60)
print("DATA QUALITY DIAGNOSIS REPORT")
print("="*60)

print("\n[1] MISSING VALUES per columns:")
print(raw_df.isnull().sum())

print(f"\n[2] DUPLICATE ROWS: {raw_df.duplicated().sum()}")

print("\n[3] DATA TYPE")
print(raw_df.dtypes)

print("\n[4] RANGE OF DATA")
print(raw_df.dtypes)

print("\n[5] UNIQUE CATEGORIES", raw_df['category'].unique())
print("\n[6] Sample customer names", raw_df['customer_name'].unique()[:8])
print("\n[7] Sample order_date values", raw_df['order_date'].unique()[:6])

DATA QUALITY DIAGNOSIS REPORT

[1] MISSING VALUES per columns:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS: 0

[3] DATA TYPE
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] RANGE OF DATA
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[5] UNIQUE CATEGORIES ['Electronics' 'Accessories' nan]

[6] Sample customer names ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' nan 'Ananya Das']

[7] Sample order_date values

In [5]:
print("NO.OF MISSING VALUES IN 'QUANTITY': ", raw_df['quantity'].isnull().sum())

NO.OF MISSING VALUES IN 'QUANTITY':  3


In [6]:

df = raw_df.copy() #Use the same for resetting df
print(f"WORKING COPY CREATED: {df.shape}")

WORKING COPY CREATED: (30, 9)


In [7]:
print("Before fixing nulls:", df.isnull().sum().sum(),"total missing values")

df['customer_name'].fillna('Unknown Customer', inplace=True)
median_qty = df['quantity'].median()
df['quantity'].fillna(median_qty, inplace=True)
print(f"Filled missing quantity with median: {median_qty}")

df['category'].fillna('Uncategorized',inplace=True)
print(f"Filled missing category with 'Uncategorized'")

df['product'].fillna('Unknown Product',inplace=True)
print(f"Filled missing product with 'Unknown Product'")

print("After fixing nulls:", df.isnull().sum().sum(),"total missing values")

Before fixing nulls: 7 total missing values
Filled missing quantity with median: 2.0
Filled missing category with 'Uncategorized'
Filled missing product with 'Unknown Product'
After fixing nulls: 0 total missing values


In [8]:

new_row_data = {
    'order_id': [1001],
    'customer_name': ['Ramesh Kumar'],
    'product': ['Laptop'],
    'category': ['Electronics'],
    'quantity': [2.0],
    'unit_price': [45000],
    'order_date': ['2024-01-05'],
    'city': ['Mumbai'],
    'sales_rep': ['Anil Sharma']
}
new_row_df = pd.DataFrame(new_row_data)
df = pd.concat([df, new_row_df]).reset_index(drop=True)
len_before = len(df)

print(f'Before deduplication: {len(df)} rows')
print(f'Duplicate rows: {df.duplicated().sum()}')
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)][['order_id','customer_name','product','order_date']].head(6))

df.drop_duplicates(inplace=True)
len_after = len(df)
print(f'\nAfter deduplication: {len(df)} rows')
print(f'Rows removed: {len_before - len_after}')

Before deduplication: 31 rows
Duplicate rows: 1

Duplicate rows:
    order_id customer_name product  order_date
0       1001  Ramesh Kumar  Laptop  2024-01-05
30      1001  Ramesh Kumar  Laptop  2024-01-05

After deduplication: 30 rows
Rows removed: 1


In [9]:
print("Sample dates before parsing:")
print(df['order_date'].head(8).tolist())

df['order_date'] = pd.to_datetime(
    df['order_date'],
    dayfirst=False,
    errors='coerce'
)

nat_count = df['order_date'].isnull().sum()
print(f"\nUnparsable dates (NaT):", nat_count)
df['year'] = df["order_date"].dt.year
df['month'] = df["order_date"].dt.month
df['month_name'] = df["order_date"].dt.strftime('%b') #(%d, %m, %Y, %A, %B, %a, %b)
df['day'] = df["order_date"].dt.day
df['day_name'] = df["order_date"].dt.strftime('%a')

df['year'].fillna('Unknown', inplace=True)
df['month'].fillna('Unknown', inplace=True)
df['month_name'].fillna('Unknown', inplace=True)
df['day'].fillna('Unknown', inplace=True)
df['day_name'].fillna('Unknown', inplace=True)

print("\nSample dates after parsing:")
df[['order_date', 'year', 'month', 'month_name','day','day_name']].head(8)

Sample dates before parsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

Unparsable dates (NaT): 2

Sample dates after parsing:


,order_date,year,month,month_name,day,day_name
0,2024-01-05,2024.0,1.0,Jan,5.0,Fri
1,2024-01-07,2024.0,1.0,Jan,7.0,Sun
2,2024-01-08,2024.0,1.0,Jan,8.0,Mon
3,2024-01-10,2024.0,1.0,Jan,10.0,Wed
4,2024-01-05,2024.0,1.0,Jan,5.0,Fri
5,NaT,Unknown,Unknown,Unknown,Unknown,Unknown
6,2024-01-12,2024.0,1.0,Jan,12.0,Fri
7,2024-01-13,2024.0,1.0,Jan,13.0,Sat


In [10]:

df['order_date'].fillna('Unknown', inplace=True)

In [11]:

df['customer_name'] = df['customer_name'].str.strip().str.title()

mask = (df['product'] == 'Keyboard') & (df['category'] == 'Electronics')
df.loc[mask, 'category'] = 'Computer Accessories'

df.head()

,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep,year,month,month_name,day,day_name
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05 00:00:00,Mumbai,Anil Sharma,2024.0,1.0,Jan,5.0,Fri
1,1002,Priya Nair,Unknown Product,Electronics,1.0,15000,2024-01-07 00:00:00,Delhi,Sunita Rao,2024.0,1.0,Jan,7.0,Sun
2,1003,Amit Verma,Keyboard,Accessories,3.0,1200,2024-01-08 00:00:00,Bangalore,Anil Sharma,2024.0,1.0,Jan,8.0,Mon
3,1004,Sunita Patel,Monitor,Electronics,2.0,22000,2024-01-10 00:00:00,Chennai,Ravi Kumar,2024.0,1.0,Jan,10.0,Wed
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05 00:00:00,Mumbai,Anil Sharma,2024.0,1.0,Jan,5.0,Fri


In [13]:
df['revenue'] = np.int64(df['quantity'] * df['unit_price'])
# df['revenue'] = (df['quantity'] * df['unit_price']).astype(int)
print(df[['customer_name', 'product', 'quantity', 'unit_price','revenue']].head(5))

print(f"\nTotal Revenue accross all orders: ₹{df['revenue'].sum():,}")

  customer_name          product  quantity  unit_price  revenue
0  Ramesh Kumar           Laptop       2.0       45000    90000
1    Priya Nair  Unknown Product       1.0       15000    15000
2    Amit Verma         Keyboard       3.0        1200     3600
3  Sunita Patel          Monitor       2.0       22000    44000
4  Ramesh Kumar           Laptop       2.0       45000    90000

Total Revenue accross all orders: ₹818,000


In [14]:
print("="*60)
print("POST CLEANING VALIDATION REPORT")
print("="*60,"\n")

print(f"[1] Original rows         : {raw_df.shape[0]}")
print(f"[2] Original columns      : {raw_df.shape[1]}")
print(f"[3] Cleaned rows          : {df.shape[0]}")
print(f"[4] Rows removed          : {len(raw_df)-len(df)}")
print(f"[5] Cleaned columns       : {df.shape[1]}")
print(f"[6] Total missing values  : {df.isnull().sum().sum()}")
print(f"[7] Duplicates            : {df.duplicated().sum()}")
print(f"[8] Date nulls            : {df['order_date'].isnull().sum()}")
print(f"[9] Revenue NaN           : {df['revenue'].isnull().sum()}")
print(f"[10]Unique categories    : {df['category'].unique()}\n")

all_clean = (
    df.isnull().sum().sum() == 0 and
    df.duplicated().sum() == 0
)

print("="*60)
print(f"DATA IS CLEAN: {all_clean}")

POST CLEANING VALIDATION REPORT

[1] Original rows         : 30
[2] Original columns      : 9
[3] Cleaned rows          : 30
[4] Rows removed          : 0
[5] Cleaned columns       : 15
[6] Total missing values  : 0
[7] Duplicates            : 0
[8] Date nulls            : 0
[9] Revenue NaN           : 0
[10]Unique categories    : ['Electronics' 'Accessories' 'Computer Accessories' 'Uncategorized']

DATA IS CLEAN: True
